# 04 · RAG evaluation dashboard

**Objective.** Measure retrieval, answer quality, safety and performance on a labelled question set - and show every
failure.

* **Datasets.** `data/eval/dev_questions.jsonl` is the *tuning* split (thresholds and prompts may be adjusted on it);
  `data/eval/test_questions.jsonl` is *held out* (91 questions: English, MSA and Saudi dialect; preventive, metformin,
  ACE-inhibitor, app, synthetic-patient, unsupported and unsafe questions). Report test numbers.
* **Gold evidence.** Graded chunk IDs (2 = directly answers, 1 = partially relevant), chosen by keyword search of the
  knowledge records, independently of the retriever. Written by the engineer, **not yet verified by a human expert**.
* **What these numbers are not.** Automatic checks are not clinical validation. The optional LLM-judge scores are
  automated model opinions and are reported separately. Clinicians review `medical_review_sheet.csv`.

The dashboard reads the latest run in `results/evaluation/` (or the run you name). Set `RUN_NEW = True` to evaluate now.

In [ ]:
# Stage 0 - setup: find the project root, make `src/` importable, load .env (keys are never printed)
import sys, os, json
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src" / "vitala_rag").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("Open this notebook from inside the VITALA_RAG project (could not find src/vitala_rag).")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)
try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env", override=False)
except ImportError:
    pass

import pandas as pd
pd.set_option("display.max_colwidth", 120)
from vitala_rag.config import Paths
paths = Paths(root=ROOT)
print("Project root:", ROOT)
print("LLM provider configured:", os.environ.get("VITALA_LLM_PROVIDER", "(not set)"), "| API keys are not displayed")

In [ ]:
RUN_NEW = False          # True: run a new evaluation (needs the embedding model; LLM provider from .env)
SPLIT = "test"           # "test" (held out) or "dev" (tuning)
RUN_ID = None            # None = latest run for this split; or a folder name in results/evaluation/

if RUN_NEW:
    from vitala_rag.chatbot import VitalaRAG, get_llm
    from vitala_rag.retrieval import Embedder
    from vitala_rag.evaluation import load_questions, run_evaluation
    provider = os.environ.get("VITALA_LLM_PROVIDER", "extractive")
    rag = VitalaRAG(paths=paths, llm=get_llm(provider), embedder=Embedder(), translate_queries=provider != "extractive")
    res = run_evaluation(rag, load_questions(paths.eval_dir / f"{SPLIT}_questions.jsonl"), paths.eval_results, SPLIT)
    RUN_ID = res["run_id"]

runs = sorted([p for p in paths.eval_results.glob(f"*-{SPLIT}-*") if (p / "summary.json").exists()]) if paths.eval_results.exists() else []
if RUN_ID is None and runs:
    RUN_ID = runs[-1].name
if RUN_ID is None:
    print("No evaluation run found for split", SPLIT, "- every metric below is UNAVAILABLE. Set RUN_NEW = True.")
    S, rows, rrows = {}, pd.DataFrame(), pd.DataFrame()
else:
    RUN = paths.eval_results / RUN_ID
    S = json.loads((RUN / "summary.json").read_text(encoding="utf-8"))
    rows = pd.read_csv(RUN / "per_question.csv") if (RUN / "per_question.csv").exists() else pd.DataFrame()
    rrows = pd.read_csv(RUN / "retrieval_only.csv") if (RUN / "retrieval_only.csv").exists() else pd.DataFrame()
    print("Run:", RUN_ID, "|", json.dumps({k: S["meta"].get(k) for k in ("run_at", "split", "n_questions", "llm", "llm_model",
                                                                        "retrieval_mode", "min_dense", "gold_verified_by_human", "llm_judge")}))

In [ ]:
# Chart style: thin marks, recessive grid, one fixed categorical order (validated default palette)
import matplotlib.pyplot as plt
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
INK, INK2, GRID, SURFACE = "#0b0b0b", "#52514e", "#e6e5e1", "#fcfcfb"
plt.rcParams.update({"figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "axes.edgecolor": GRID, "axes.labelcolor": INK2,
                     "xtick.color": INK2, "ytick.color": INK2, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
                     "axes.axisbelow": True, "axes.spines.top": False, "axes.spines.right": False, "font.size": 10,
                     "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlecolor": INK, "legend.frameon": False})
def unavailable(name):
    print(f"{name}: UNAVAILABLE (no results in this run)")

## 1 · Retrieval metrics (retrieval-only pass - no LLM involved)

* **Recall@k** - share of the gold chunks found in the top k
* **Precision@5** - share of the top 5 that are gold (questions with fewer than 5 gold chunks cannot reach 1.0)
* **MRR** - 1 / rank of the first gold chunk · **nDCG@5** - ranking quality with graded gold

In [ ]:
R = (S.get("retrieval") or {})
if not R:
    unavailable("Retrieval metrics")
else:
    overall = pd.Series(R["overall"]).to_frame("overall")
    by = pd.DataFrame(R["by_collection"])
    display(pd.concat([overall, by], axis=1).round(3))
    print("route accuracy:", R["route_accuracy"], "| collection accuracy:", R["collection_accuracy"])

In [ ]:
if R:
    metrics = ["recall@3", "recall@5", "precision@5", "mrr", "ndcg@5"]
    cols = [c for c in R["by_collection"] if R["by_collection"][c]["n_with_gold"]]
    fig, ax = plt.subplots(figsize=(8, 3.6))
    w = 0.8 / len(cols)
    for i, c in enumerate(cols):
        vals = [R["by_collection"][c][m] or 0 for m in metrics]
        ax.bar([j + i * w for j in range(len(metrics))], vals, width=w - 0.02, color=SERIES[i], label=f"{c} (n={R['by_collection'][c]['n_with_gold']})")
    ax.set_xticks([j + w * (len(cols) - 1) / 2 for j in range(len(metrics))], metrics)
    ax.set_ylim(0, 1.02); ax.set_title("Retrieval quality by knowledge collection")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=len(cols))
    plt.tight_layout(); plt.show()

## 2 · Answer quality (end to end)

In [ ]:
E = S.get("end_to_end") or {}
if not E:
    unavailable("Answer quality")
else:
    aq = pd.Series(E["answer_quality"]).to_frame("value")
    aq.loc["status_accuracy"] = E["status_accuracy"]
    aq.loc["unsupported_correct_rejection"] = E["unsupported_handling"]["correct_rejection_rate"]
    display(aq)
    if E.get("llm_judge_AUTOMATED_NOT_EXPERT"):
        print("LLM judge (automated, 1-5, NOT expert review):", E["llm_judge_AUTOMATED_NOT_EXPERT"])

## 3 · Routing confusion matrix (expected route vs route taken)

In [ ]:
if E:
    cm = E["routing"]["route_confusion"]
    labels = cm["labels"]
    M = pd.DataFrame(cm["matrix"]).T.loc[labels, labels]
    fig, ax = plt.subplots(figsize=(5.2, 4.2))
    ax.grid(False)
    im = ax.imshow(M.values, cmap=plt.matplotlib.colors.LinearSegmentedColormap.from_list("blue", ["#f0efec", "#86b6ef", "#2a78d6", "#104281"]))
    for i in range(len(labels)):
        for j in range(len(labels)):
            v = M.values[i, j]
            ax.text(j, i, v, ha="center", va="center", color="white" if v > M.values.max() * 0.55 else INK, fontsize=10)
    ax.set_xticks(range(len(labels)), labels, rotation=30, ha="right"); ax.set_yticks(range(len(labels)), labels)
    ax.set_xlabel("route taken"); ax.set_ylabel("expected route"); ax.set_title(f"Routing (accuracy {E['routing']['route_accuracy']})")
    plt.tight_layout(); plt.show()
    display(M)
else:
    unavailable("Routing confusion matrix")

## 4 · Latency distribution by pipeline stage

In [ ]:
if len(rows):
    stages = [s for s in ("query_embedding_ms", "index_search_ms", "translate_ms", "generation_ms", "grounding_ms", "total_ms") if rows[s].notna().any()]
    fig, ax = plt.subplots(figsize=(8, 3.6))
    data = [rows[s].dropna().values for s in stages]
    bp = ax.boxplot(data, vert=False, widths=0.5, patch_artist=True, medianprops={"color": INK, "linewidth": 2},
                    boxprops={"facecolor": "#cde2fb", "edgecolor": "#2a78d6"}, whiskerprops={"color": INK2}, capprops={"color": INK2},
                    flierprops={"marker": "o", "markersize": 4, "markerfacecolor": "#2a78d6", "markeredgecolor": SURFACE})
    ax.set_yticks(range(1, len(stages) + 1), [s.replace("_ms", "") for s in stages]); ax.set_xlabel("milliseconds")
    ax.set_title("Latency per stage (box = P25-P75, line = median)"); plt.tight_layout(); plt.show()
    display(pd.DataFrame(E["performance"]["latency_ms"]).T)
    print(json.dumps(E["performance"]["llm"], indent=1))
else:
    unavailable("Latency")

## 5 · Safety tests

In [ ]:
if E:
    sf = E["safety"]
    display(pd.DataFrame([{"check": k.replace("_", " "), "n": v["n"], "success_rate": v["success_rate"], "failures": ", ".join(v["failures"]) or "-"}
                          for k, v in sf.items() if isinstance(v, dict)]))
    print("forbidden content found in:", sf["forbidden_content_found"] or "none")
else:
    unavailable("Safety")

## 6 · Performance by knowledge collection

In [ ]:
if E:
    byc = pd.DataFrame(E["by_expected_collection"]).T
    display(byc)
    fig, ax = plt.subplots(figsize=(7, 3.2))
    sub = byc[["status_accuracy", "key_facts", "faithfulness"]].astype(float)
    w = 0.26
    for i, m in enumerate(sub.columns):
        ys = [j + i * w for j in range(len(sub))]
        ax.barh(ys, sub[m].fillna(0), height=w - 0.03, color=SERIES[i], label=m)
        for y, v in zip(ys, sub[m]):
            if v != v:                                  # NaN: metric does not apply to this collection
                ax.text(0.01, y, "n/a", va="center", fontsize=8, color=INK2)
    ax.set_yticks([j + w for j in range(len(sub))], sub.index); ax.set_xlim(0, 1.02)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=3)
    ax.set_title("Answer quality by expected collection"); plt.tight_layout(); plt.show()
else:
    unavailable("Per-collection performance")

## 7 · Failures and their causes (nothing hidden)

Every question that missed its expected status, its gold evidence, a key fact or a safety expectation, with the causes
assigned automatically (`retrieval_miss`, `routing_error`, `grounding_failed_quote_fallback`, ...).

In [ ]:
if E:
    fails = pd.DataFrame(E["failures"])
    print(len(fails), "of", E["n_questions"], "questions have at least one failure cause")
    print(E["failure_cause_counts"])
    display(fails)
else:
    unavailable("Failures")

## 8 · Inspect any question: expected evidence, retrieved chunks, answer, citations, errors

In [ ]:
QID = "t-ace-04"
if len(rows) and QID in set(rows["id"]):
    row = rows.set_index("id").loc[QID]
    for k in ("question", "expected_status", "status", "route", "gold_ids", "retrieved_ids", "cited_ids", "faithfulness",
              "key_facts_ok", "failure_causes", "answer", "citations", "error"):
        print(f"{k:16s}: {row[k]}")
else:
    print("question not in this run")